# Análisis Inferencial y Multivariado

Este notebook contiene dos pruebas inferenciales clave para el proyecto Ganado Saludable:
1. **Correlación Cruzada Zoonótica**: Tuberculosis Bovina vs Tuberculosis Humana.
2. **ANOVA**: Riesgo de Salmonella según Canal de Venta.

## 1. Correlación Cruzada (Zoonosis)

El objetivo es demostrar que los estados con mayor carga de tuberculosis bovina (datos de SENASICA) presentan también mayores índices de tuberculosis humana (datos de morbilidad DGE).

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Cargar datos de Salud Humana (DGE)
dge_df = pd.read_csv('../data/processed/dge_morbilidad_clean.csv')
# Filtrar códigos CIE-10 correspondientes a Tuberculosis (A15-A19)
dge_tb = dge_df[dge_df['cie_10'].str.startswith('A1', na=False)]
# Agrupar casos humanos por estado
casos_humanos = dge_tb.groupby('estado')['casos'].sum().reset_index()
casos_humanos.rename(columns={'casos': 'casos_humanos_tb'}, inplace=True)

# 2. Cargar datos de Sanidad Animal (SENASICA)
sen_df = pd.read_csv('../data/processed/senasica_cuarentenas_clean.csv')
# Agrupar animales en cuarentena por estado
casos_animales = sen_df.groupby('estado')['num_animales'].sum().reset_index()
casos_animales.rename(columns={'num_animales': 'animales_cuarentena_tb'}, inplace=True)

# 3. Unir ambos datasets
df_corr = pd.merge(casos_humanos, casos_animales, on='estado', how='inner')
df_corr.head()


In [ ]:
# 4. Calcular Correlación de Pearson
corr, p_val = stats.pearsonr(df_corr['animales_cuarentena_tb'], df_corr['casos_humanos_tb'])

print("--- CORRELACIÓN TB BOVINA VS TB HUMANA ---")
print(f"Coeficiente de Correlación (r): {corr:.4f}")
print(f"P-value: {p_val:.4e}")

if p_val < 0.05:
    print("\nConclusión: Existe una correlación estadísticamente significativa.")
else:
    print("\nConclusión: No existe evidencia suficiente para afirmar correlación.")

# 5. Visualización
plt.figure(figsize=(8, 5))
sns.regplot(x='animales_cuarentena_tb', y='casos_humanos_tb', data=df_corr, color='crimson')
plt.title('Correlación: TB Bovina vs TB Humana por Estado')
plt.xlabel('Animales en Cuarentena (SENASICA)')
plt.ylabel('Casos de TB Humana (DGE)')
plt.show()


## 2. Análisis de Varianza (ANOVA): Canales de Venta

Vamos a comprobar si el canal de comercialización influye estadísticamente en el riesgo de encontrar Salmonella spp., utilizando los porcentajes documentados en nuestro V2.md.

In [ ]:
# Prevalencias conocidas
prevalencias = {
    'Supermercados': 0.013,  # 1.3%
    'Carnicerías': 0.084,    # 8.4%
    'Tianguis': 0.136,       # 13.6%
    'Mercados Municipales': 0.223  # 22.3%
}

# Simularemos 1000 muestras para cada canal para realizar el test ANOVA
n_muestras = 1000
np.random.seed(42)

datos = []
for canal, prev in prevalencias.items():
    muestras = np.random.binomial(1, prev, n_muestras)
    for m in muestras:
        datos.append({'Canal': canal, 'Salmonella_Positiva': m})

df_anova = pd.DataFrame(datos)

# Separar grupos
grupos = [df_anova[df_anova['Canal'] == c]['Salmonella_Positiva'] for c in prevalencias.keys()]

# Aplicar ANOVA (f_oneway)
f_stat, p_value = stats.f_oneway(*grupos)

print("--- RESULTADOS ANOVA: CANALES DE VENTA ---")
print(f"Estadístico F: {f_stat:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    print("\nConclusión: RECHAZAMOS la hipótesis nula. El canal de venta SÍ importa para la inocuidad.")
else:
    print("\nConclusión: NO RECHAZAMOS la hipótesis nula.")
    
# Gráfica
plt.figure(figsize=(9, 5))
sns.barplot(x='Canal', y='Salmonella_Positiva', data=df_anova, errorbar=('ci', 95), capsize=0.1, palette='Oranges', hue='Canal', legend=False)
plt.title('Prevalencia de Salmonella por Canal (Intervalos de Confianza 95%)')
plt.ylabel('Tasa de Prevalencia')
plt.show()
